# FHIR Silver transformation — SCD Type 2

This notebook validates, cleans and deduplicates Bronze resource versions, then rebuilds one Silver SCD Type 2 table per required resource. It uses Spark—not Dataflows Gen2—for JSON/table processing. Bronze retains distinct content versions; Silver derives validity intervals from `meta.lastUpdated` when available, with extraction/save timestamps as fallback.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

RESOURCE_TYPES = ["Patient", "Encounter", "Observation", "Condition"]
spark.conf.set("spark.sql.session.timeZone", "UTC")

def table_exists(name):
    return spark.catalog.tableExists(name)

run_ts = datetime.now(timezone.utc).isoformat()
summary = []

In [ ]:
for resource_type in RESOURCE_TYPES:
    bronze_table = f"bronze.{resource_type}"
    silver_table = f"silver.{resource_type}"

    if not table_exists(bronze_table):
        raise RuntimeError(f"Missing {bronze_table}. Run ingestion for {resource_type} first.")

    bronze = spark.table(bronze_table)

    # Remove malformed rows from Silver but leave their original API response in Raw.
    clean = bronze.filter(
        F.col("resource_id").isNotNull()
        & F.col("resource_json").isNotNull()
        & F.col("content_hash").isNotNull()
    )

    # Repeated incremental overlap can fetch a resource again; identical hashes are one version.
    dedupe_window = Window.partitionBy("resource_type", "resource_id", "content_hash").orderBy(
        F.to_timestamp("saved_timestamp").asc_nulls_last(),
        F.to_timestamp("extraction_timestamp").asc_nulls_last()
    )
    clean = (clean.withColumn("_copy_rank", F.row_number().over(dedupe_window))
                  .filter(F.col("_copy_rank") == 1)
                  .drop("_copy_rank"))

    # Source timestamp is authoritative. Fall back only when it is missing.
    clean = (clean.withColumn("_source_ts", F.to_timestamp("source_last_updated"))
                  .withColumn("_extract_ts", F.to_timestamp("extraction_timestamp"))
                  .withColumn("_save_ts", F.to_timestamp("saved_timestamp"))
                  .withColumn("_effective_ts", F.coalesce(F.col("_source_ts"), F.col("_extract_ts"), F.col("_save_ts"))))

    version_window = Window.partitionBy("resource_type", "resource_id").orderBy(
        F.col("_effective_ts").asc_nulls_first(),
        F.col("_save_ts").asc_nulls_last(),
        F.col("content_hash").asc()
    )
    silver = (clean.withColumn("valid_from", F.col("_effective_ts"))
                   .withColumn("valid_to", F.lead(F.col("_effective_ts")).over(version_window))
                   .withColumn("is_current", F.col("valid_to").isNull())
                   .withColumn("scd2_rebuilt_timestamp", F.lit(run_ts).cast("timestamp"))
                   .drop("_source_ts", "_extract_ts", "_save_ts", "_effective_ts"))

    # Deterministic rebuild from every observed Bronze version; historical versions remain in Silver.
    (silver.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(silver_table))

    total_versions = spark.table(silver_table).count()
    current_versions = spark.table(silver_table).filter(F.col("is_current") == True).count()
    summary.append({"resource": resource_type, "bronze_rows": bronze.count(), "silver_versions": total_versions,
                    "current_rows": current_versions, "table": silver_table})

print("Silver SCD Type 2 rebuild summary:")
for item in summary:
    print(item)

## SCD Type 2 implementation note

Silver is reconstructed from all distinct content versions observed in Bronze at each run. This makes reruns deterministic and retains older versions, while setting the final observed version per resource to `is_current = true` and closing prior intervals with `valid_to`. If the public API has not changed a resource during the observation period, multiple versions will not appear for that resource; the logic will create history when different content hashes are ingested.